# Tutorial on Model Evaluation

This tutorial is part of the course Foundations of Data Analysis (summer term 2023) at the University of Vienna.

## Model Validation of a Classifier (Logistic Regression)

We will go through the principles of model validation using a synthetic dataset with a discrete target (four classes).
Note: Explanations come from the lecture on model validation.

In [ ]:
# Generate Data:

import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
X,y = make_classification(n_samples=10000, n_features=10, n_redundant=2, n_informative=8, n_classes=4, class_sep=2, random_state=42)

df = pd.DataFrame(X, columns=[f"X{i}" for i in range(1,11,1)])
df["target"] = y

# good practice to have a quick glance at the data (typically, you make some plots for descriptive data analysis
# or visual checks of potential model assumptions, e.g. normality of the data when using, for example, lda)
df.head()

In the lecture you have learned that using train data to estimate the model risk (or in general evaluate model performance) leads to too optimistic results. Please take a minute to think about/recall why this is the case. 

Now, recall the theorem from the model validation lecture.

Theorem: Let $h$ be a hypothesis and assume that the loss function is in $[0,1]$. Then for every $\delta \in [0,1]$ we have that with probability of at least $1-\delta$ over the choice of a test set of size $m_v$ we have: \
\begin{equation}
|L_v(h) - L_D(h)| \leq \sqrt{\frac{\log{\frac{2}{\delta}}}{2m_v}}
\end{equation} \
This means that if $m_v$ grows large, the risk estimated on the test set ($L_v(h)$) of this size comes closer to the true risk ($L_D(h)$). Note that the theorem only relies on the sample size of the test set and not the VC-dimension. Hence, we want to split our data into a training set for model fitting (choosing the ERM hypothesis from a hypothesis class) and a test set for model evaluation.

In [ ]:
from sklearn.model_selection import train_test_split # function to split data

X = df.drop("target", axis=1)
y = df["target"]

# split data

Having split the data, fit a model on the training data and evaluate it on the test data, which is a first step in model validation.

In [1]:
from sklearn.linear_model import LogisticRegression # the model
from sklearn.metrics import accuracy_score, zero_one_loss

# model + evaluation

## k-fold Cross Validation

Recall that the bound from the Theorem above has to be multiplied by the number of tests if you use same test set on different models. So what to do if you have to conduct many tests? This is the case if you want to find the best models from many hypothesis classes and then choose the best performing out of all these models.

In this case, you can use k-fold cross validation (from training set) to choose a model. Cross validation helps to evaluate risk-minimising hypotheses from a large number of different hypothesis classes that, for example, can vary by a hyperparameter $\lambda$ (hence, we will denote the hypothesis classes $\mathcal{H}_{\lambda}$). It works as outlined below:

- Let $S$ be your training data. Split it into $K$ (equally large) subsets $S_1, ...,S_K$.
- Let $\mathcal{I} = \{1,...,K \}$ be the index set of these subsets
- For every hypothesis class $\mathcal{H}_{\lambda}$:
    * for i in $\mathcal{I}$:
        1. Find risk minimising hypothesis using $S \setminus S_i = \bigcup_{j \in \mathcal{I}\setminus i} S_j$; denote hypothesis $h_s^i$
        2. Let $V_i$ be the currently held out $S_i$ in step $i$ to stress that it is used for validation, calculate model risk $L_{V_i}(h_s^i)$
    * Calculate average model risk: $\hat{L}_D(h) = \frac{1}{K} \sum_{i=1}^K L_{V_i}(h_s^i)$
- Choose the hypthesis class with the lowest average model risk.
- Once decided on a hypthesis class, use the whole $S$ to find the ERM model from this class.
- You can now use the test set to estimate the model's performance.

The code below performs this with $k=10$, which is a standard choice.

In [ ]:
# CV from scratch
k = 10 # choice of k (typically 10)

'''Define several hypothesis classes to choose a hypothesis from. Here, the parameter C in LogisticRegression() re-
presents a shift in the hyopthesis class (that's all you need to know at this point); C corresponds to lambda.'''

C = [i/10 for i in range(1,11,1)] # [0.1, 0,2, ..., 1]; i.e., 10 different hypothesis classes

# the model looks like this: clf_c = LogisticRegression(C=c, penalty="l2") -> find a good c



Below we use an implementation from the Python package scikit-learn that makes life much easier. Note, there are functions that implement cross validation in one line (some times for a specific model), however, we go through the steps explicitly for better understanding of the steps.

In [ ]:
# imoport the cross_val_score from the model_selection module from scikit-learn
from sklearn.model_selection import cross_val_score

# see above
C = [i/10 for i in range(1,11,1)] # [0.1, 0,2, ..., 1]; i.e., 10 different hypothesis classes


Final note on cross validation: Recall the example of CV with a linear model for a binary classification task from the lecture (Lecture on model validation at min. 25:00 onwards). The example highlights that CV can fail to be a good estimate for your model performance. Hence, it is used as model selection technique only. After having decided on the best model, you have to test it on unseen data to estimate its performance. But what if performance then is not satisfactory?

There are two types of errors that can lead to bad model performance:

1. Large approximation error $L_D(h^*)$, i.e. a large error for the optimal hypothesis given the hypopthesis class
    * Can potentially by solved by choosing different/enlarging the hypopthesis class
    * or by adding more features, which essentially is choosing a different/enlarging the hypothesis class (think about why this is the case, e.g. highlighting it with linear regression) 
 <br>
 <br>
    
2. Large estimation error $L_D(h_S) - L_D(h^*)$, i.e. the ERM hypothesis $h_S$ performs (much) worse than the optimal hypothesis from the given class
    * Can potentially be solved by increasing the sample size (size of training set),
    * reducing the model complexity,
    * or using a different learning algorithm

How to find out, which error (1. or 2.) dominates? Recall from the lecture that we want to learn something about $L_D(h_S)$, i.e. the true risk of the 'chosen' model. But we have no access to it. Hence, rewrite it to:
\begin{equation}
    L_D(h_S) = L_D(h_S) - L_v(h_S) + L_v(h_S) - L_S(h_S) + L_S(h_S)
\end{equation}

Since $L_D(h_S) - L_v(h_S)$ quickly goes to zero for increasing sample size of the validation set, we have to terms that can drive up $L_D(h_S)$:
1. $L_v(h_S) - L_S(h_S)$: Difference between estimated model risk on validation set and on training set
    * would mean large estimation error e.g. because of overfitting to the data
    <br>
    <br>
    
2. $L_S(h_S)$: Estimated model risk on training set
    * high approximation error, i.e. no good hypothesis in hypothesis class


How to find out, what is the dominating reason behind poor performance?

Draw a learning curve:
- x-axis: no. of samples in training set
- y-axis: estimated risk (or other evaluation measure)

In [ ]:
# Learning curve from scratch
import matplotlib.pyplot as plt # standard library for plotting

# A learning curve plots the error on the training data and a validation set for different sample sizes 
# (used during training)

